# 06 — Run Packed Weights on an Input Vector

**Holonomy** here means: take each packed 8×8 operator in order and apply them to an input vector — like sending a small list of numbers through a stack of compressed layers.

Rough idea:

`output = (layer_N ∘ … ∘ layer_1)(input)`

This notebook packs the example weights, runs `crankl holonomy`, and prints:

- input vector
- output vector
- element-by-element change
- a few summary stats (norms, MSE vs a calibration target)

This shows Crankl’s forward API. It is **not** a full neural-network runtime.

In [ ]:
from pathlib import Path
import sys
import numpy as np

notebook_dir = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").exists() else Path.cwd()
sys.path.insert(0, str(notebook_dir)) if str(notebook_dir) not in sys.path else None

from crankl_demo import ARTIFACT_DIR, prepare_demo_files, run_cli

np.set_printoptions(precision=5, suppress=True)
demo = prepare_demo_files()
archive_path = ARTIFACT_DIR / "holonomy_example.crank"
output_path = ARTIFACT_DIR / "holonomy_output.f32"

run_cli("pack", "--input", demo.source_path, "-o", archive_path)
run_cli("holonomy", "--input", archive_path,
        "--vector", demo.calibration_x_path, "-o", output_path)

forward_output = np.fromfile(output_path, dtype=np.float32)
print("Input vector: ", demo.calibration_x)
print("Output vector:", forward_output)
print("Change:       ", forward_output - demo.calibration_x)

## Summarize and compare

Output length matches input length. We print vector sizes (L2 norms) and mean squared error against the demo calibration target `Y`.

That MSE is just an illustration of how a caller might score the output — similar in spirit to Crankl’s holonomy calibration loss.

In [ ]:
input_norm = np.linalg.norm(demo.calibration_x)
output_norm = np.linalg.norm(forward_output)
change_norm = np.linalg.norm(forward_output - demo.calibration_x)
calibration_mse = np.mean((forward_output - demo.calibration_y) ** 2)

print(f"Input L2 norm:             {input_norm:.6f}")
print(f"Output L2 norm:            {output_norm:.6f}")
print(f"Input-to-output change:    {change_norm:.6f}")
print(f"MSE vs calibration target: {calibration_mse:.6f}")
print("Expected calibration Y:  ", demo.calibration_y)

## Peek at the archive behind the transform

`inspect --json` ties the output to the packed file’s slot count, gamma, trit density, entropy, energy, and beta1 proxy.

For archives created by plain `pack`, gamma defaults to **1**.

In [ ]:
archive_report = run_cli("inspect", archive_path, "--json", expect_json=True)
print("\nArchive metrics used for this transform:")
for name, value in archive_report["metrics"].items():
    print(f"  {name:20s} {value}")